The code here is for paid AI models such as OpenAI, Claude etc. For the free models we will use Output Parser.

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task = "text-generation"
)

model = ChatHuggingFace(llm=llm)
result = model.invoke("What is the capital of Pakistan")
result

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

AIMessage(content='<|user|>\nWhat is the capital of Pakistan</s>\n<|assistant|>\nThe capital of Pakistan is Islamabad.', additional_kwargs={}, response_metadata={}, id='lc_run--019d67b3-d154-73e3-a549-175095db9d5f-0', tool_calls=[], invalid_tool_calls=[])

In [ ]:
# TypedDict with_structued_output
from typing import TypedDict

class Review(TypedDict):

  summary : str
  sentiment : str

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""The hardware is great, but the software feels bloated.
                        There are too many pre-installed apps that I can't remove.
                        Also, the UI looks outdated compared to other brands.
                        Hoping for a software update to fix this.""")

print(result)

None


In [ ]:
# There my be ambiguity what to generate sometimes then we use annoted dict
from typing import TypedDict, Annotated, Optional, Literal

class Review(TypedDict):

  main_theme: Annotated[list[str], "Write down the key theme discussed in the review"]
  summary : Annotated[str, "A brief Summary of the review"]
  sentiment : Annotated[Literal['pos', 'neg'], "A sentiment of the review either positive, negative, or neutral"]
  pros : Annotated[Optional[list[str]], "Write down all the pros in a list"]
  cons : Annotated[Optional[list[str]], "Write down all the cons in a list"]


structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""The hardware is great, but the software feels bloated.
                        There are too many pre-installed apps that I can't remove.
                        Also, the UI looks outdated compared to other brands.
                        Hoping for a software update to fix this.""")

print(result)

In [ ]:
#Pydantic also gives us data validation option

from pydantic import BaseModel, EmailStr, Field
from typing import Optional

class Student(BaseModel):

  name: str
  # name: str = "John Doe"  Default value
  age : Optional[int] = None
  email: EmailStr  # To Validate Emails
  cpga: float = Field(gt=0, lt=10) # Field is used to add constraint. Also use Field to add description

new_student = {"name","John"}

print(**new_student)

None


In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Review(BaseModel):


  main_theme: list[str] = Field(description= "Write down the key theme discussed in the review")
  summary : str = Field(description= "A brief Summary of the review")
  sentiment : Literal['pos', 'neg'] =  Field(description= "A sentiment of the review either positive, negative, or neutral")
  pros : Optional[list[str]] = Field(default = None, description= "Write down all the pros in a list")
  cons : Optional[list[str]] = Field(default = None, description="Write down all the cons in a list")


structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""The hardware is great, but the software feels bloated.
                        There are too many pre-installed apps that I can't remove.
                        Also, the UI looks outdated compared to other brands.
                        Hoping for a software update to fix this.""")

print(result)

In [ ]:
# Write the given code in json file for json_schema
{
    "title" : "student",
    "description" : "schema for students",
    "type" : "object",
    "property" : {
        "name": "string",
        "age": "integer"
    },
    "required" : ['name'], # Others are optional
}